# Data audit & baseline — antimicrobial demand (HU-UFPB)
**MLS 2026.2 · Team F.** This notebook runs end to end and regenerates the
processed base. It uses the `huforecast` package (installed via `uv`/`pip -e .`),
so there are no absolute paths and no manual steps.

Pipeline layers: `ingestion → processing → features → modeling`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import huforecast as hf
from huforecast import config
from huforecast.features.timeseries import total_monthly_demand

## 1. Ingestion — read the raw treatment workbook (untouched)

In [2]:
raw = hf.load_raw_treatments()
print("Raw treatments:", raw.shape)
raw.head()

Raw treatments: (9941, 12)


,SETOR,SITUAÇÃO,DIAS,POSO,QTD/DIA,TOTAL,RESTA,SUSPENDER,INÍCIO,TÉRMINO,ANTIMICROBIANO,QTD
0,UTI - ADULTO,ENCERRADO,11.0,12/12,4.0,44,NaN,,2025-03-06,2025-03-16,"POLIMIXINA B SULFATO, 500.000 UI, PÓ LIÓFILO",FIM
1,UTI - ADULTO,ENCERRADO,11.0,24/24,2.0,22,NaN,NaN,2025-03-06,2025-03-16,"AMICACINA SULFATO, 250 MG/ML, SOLUÇÃO INJETÁVE...",FIM
2,UTI - ADULTO,ENCERRADO,11.0,24/24,1.0,11,NaN,,2025-03-06,2025-03-16,"MICAFUNGINA, 50 MG, PÓ LIÓFILO",FIM
3,CLÍNICA MÉDICA,ENCERRADO,15.0,8/8,3.0,45,NaN,NaN,2025-03-06,2025-03-20,"MEROPENEM, 1 G, PÓ LIÓFILO",FIM
4,UTI - ADULTO,SUSPENSO,15.0,12/12,4.0,60,NaN,SIM,2025-03-06,2025-03-20,"VANCOMICINA CLORIDRATO, 500 MG, PÓ LIÓFILO",STOP


## 2. Processing — clean, type, deduplicate
Decisions: drop near-empty columns (`qtd_resta`, `qtd_dispensada`), turn
`suspender` into a binary flag, standardize categories, drop exact duplicates
and rows with no start date.

In [3]:
clean = hf.clean_treatments(raw)
print("Clean base:", clean.shape, "| removed:", len(raw) - len(clean))
print("\nMissing (%):")
print((clean.isna().mean()*100).round(1).sort_values(ascending=False))
clean.head()

Clean base: (7261, 13) | removed: 2680

Missing (%):
suspender          78.4
qtd_dia             1.2
setor               0.4
posologia           0.2
qtd_total           0.1
situacao            0.0
dias_tratamento     0.0
data_inicio         0.0
data_termino        0.0
antimicrobiano      0.0
suspender_flag      0.0
principio_ativo     0.0
mes_inicio          0.0
dtype: float64


,setor,situacao,dias_tratamento,posologia,qtd_dia,qtd_total,suspender,data_inicio,data_termino,antimicrobiano,suspender_flag,principio_ativo,mes_inicio
0,UTI - ADULTO,ENCERRADO,11.0,12/12,4.0,44.0,,2025-03-06,2025-03-16,"POLIMIXINA B SULFATO, 500.000 UI, PÓ LIÓFILO",1,POLIMIXINA B SULFATO,2025-03
1,UTI - ADULTO,ENCERRADO,11.0,24/24,2.0,22.0,NaN,2025-03-06,2025-03-16,"AMICACINA SULFATO, 250 MG/ML, SOLUÇÃO INJETÁVE...",0,AMICACINA SULFATO,2025-03
2,UTI - ADULTO,ENCERRADO,11.0,24/24,1.0,11.0,,2025-03-06,2025-03-16,"MICAFUNGINA, 50 MG, PÓ LIÓFILO",1,MICAFUNGINA,2025-03
3,CLÍNICA MÉDICA,ENCERRADO,15.0,8/8,3.0,45.0,NaN,2025-03-06,2025-03-20,"MEROPENEM, 1 G, PÓ LIÓFILO",0,MEROPENEM,2025-03
4,UTI - ADULTO,SUSPENSO,15.0,12/12,4.0,60.0,SIM,2025-03-06,2025-03-20,"VANCOMICINA CLORIDRATO, 500 MG, PÓ LIÓFILO",1,VANCOMICINA CLORIDRATO,2025-03


## 3. Overview — what the data says

In [4]:
print("Period:", clean["data_inicio"].min().date(), "→", clean["data_inicio"].max().date())
print("Antimicrobials (active ingredient):", clean["principio_ativo"].nunique())
print("Sectors:", clean["setor"].nunique())
print("\nTreatment status:")
print(clean["situacao"].value_counts(dropna=False))

Period: 2025-03-06 → 2026-10-22
Antimicrobials (active ingredient): 32
Sectors: 17

Treatment status:
situacao
ENCERRADO        5677
SUSPENSO         1473
EM TRATAMENTO     111
Name: count, dtype: int64


## 4. Features — monthly demand time series

In [5]:
series = hf.monthly_demand_by_drug(clean)
m = total_monthly_demand(series)
print("Months in series:", len(m))
print(m.to_string(index=False))

Months in series: 20
mes_inicio  qtd_total
   2025-03    13022.0
   2025-04    11089.0
   2025-05    15024.0
   2025-06    13031.0
   2025-07    14205.5
   2025-08    14960.0
   2025-09    13680.0
   2025-10    14494.0
   2025-11    13724.0
   2025-12    14563.0
   2026-01    15269.0
   2026-02    14594.0
   2026-03    12380.0
   2026-04    15319.0
   2026-05    17622.0
   2026-06    15652.0
   2026-07    17007.0
   2026-08    12930.0
   2026-09    14578.0
   2026-10     8419.0


## 5. Data quality note — incomplete last month
Collection happened on 2026-10-22, so `2026-10` is partial and is excluded from
modeling to avoid biasing the metric.

In [6]:
print("Last month in series:", m["mes_inicio"].iloc[-1], "→ dropped (partial collection)")

Last month in series: 2026-10 → dropped (partial collection)


## 6. Modeling — temporal split + baselines
The decision forecasts next month's demand, so the split is **temporal**:
train on earlier months, test on the latest ones (no leakage).

In [7]:
res = hf.run_baselines(series)
import json
print(json.dumps(res, indent=2, default=str, ensure_ascii=False))

{
  "n_months": 19,
  "train_span": [
    "2025-03",
    "2026-05",
    15
  ],
  "test_span": [
    "2026-06",
    "2026-09",
    4
  ],
  "baseline_trivial_mean": {
    "MAE": 1477.5333333333338,
    "MAPE": 9.55366429786887
  },
  "baseline_operational_persistence": {
    "MAE": 2262.5,
    "MAPE": 15.847396691081473
  }
}


## 7. Baseline — the number every model must beat

In [8]:
tr = res["baseline_trivial_mean"]; op = res["baseline_operational_persistence"]
print("Trivial  (train mean):   MAPE = %.1f%% | MAE = %.0f" % (tr["MAPE"], tr["MAE"]))
print("Operational (persist.):  MAPE = %.1f%% | MAE = %.0f" % (op["MAPE"], op["MAE"]))
print("\n>>> SEMESTER REFERENCE: MAPE ~9.6%% (mean). Any model must beat it.")

Trivial  (train mean):   MAPE = 9.6% | MAE = 1478
Operational (persist.):  MAPE = 15.8% | MAE = 2262

>>> SEMESTER REFERENCE: MAPE ~9.6%% (mean). Any model must beat it.


## 8. Persist the processed base

In [9]:
config.ensure_dirs()
clean.to_parquet(config.PROCESSED_DIR / "tratamentos_antimicrobianos.parquet", index=False)
series.to_parquet(config.PROCESSED_DIR / "serie_mensal.parquet", index=False)
print("Processed base saved to", config.PROCESSED_DIR)
print("\nNotebook executed end to end successfully.")

Processed base saved to /Users/brennohdev/development/college/machine-learning-2026.2/data/processed

Notebook executed end to end successfully.
